# 03 - Train a DQN Model Online

This notebook shows the online training workflow in Mouse Core. Instead of loading a fixed dataset, it keeps live environments in the loop:

1. Build a train `GroupEnv`.
2. For each of `NUM_CYCLES` cycles: collect `ROLLOUT_STEPS` env steps, then run `TRAIN_STEPS` optimizer updates.
3. Append transitions into in-memory `Datastore` replay streams.
4. Sample those streams with `DataLoader`.
5. Train with `DqnObjective`.

Online and offline training use the same row format, model interface, datastores, dataloader, and objective. Only the source of rows changes.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb`.


In [ ]:
import torch
import numpy as np

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core import AdamW, AdamWFp32
from mouse_core.models.kv_policy import (
    cache_needs_rebuild,
    rebuild_starts,
    resolve_cache_bounds,
)

from mouse_core.data import (
    DataLoader,
    Datastore,
    NumericTokenizer,
    pack_token_batch,
)
from mouse_core.models import Model, PolyakAverager, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead
from mouse_core.objectives import DqnObjective


MODEL_ID = "mouse-example-model-online"       # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
MAX_STEPS_PER_EPISODE = 30                    # max steps per episode
MAX_EPISODES_PER_TASK = 20                    # max episodes per task
NUM_ENVS = 4                                  # number of environment streams in the GroupEnv

SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step

NUM_CYCLES = 2                               # outer rollout/train cycles
ROLLOUT_STEPS = 64                           # env steps per cycle (passed to run_rollout)
TRAIN_STEPS = 20                              # optimizer updates per cycle (passed to run_train)
POLYAK_TAU = 0.0005                           # Q-head Polyak coefficient (encoder/backbone stay online)

LEARNING_STARTS = 64                          # replay rows collected before the first optimizer update
EXPLORATION_ENDS = 1_000                      # env-step horizon for epsilon decay

# Rollout rows are appended to datastores and become replay samples after loader.refresh().


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Build Environment

Online training uses the same `EnvConfig` and `make_group_env` pattern as data collection. Build a **train** `GroupEnv` for rollouts.

The group environment steps all configured instances together and returns one output dictionary per instance. Keep row fields consistent with the model and objective: this notebook consumes `action`, `observation`, `reward`, `episode_done`, and `task_done`; the DQN objective uses `episode_done` and `task_done` to decide whether bootstrapping should continue across a boundary.


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        seed=i,
        episodes_per_task=MAX_EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # forwarded to the environment at task reset
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_STEPS_PER_EPISODE,
            "map_seed": i,
            "slippery_success_rate": 1.0,  # environment-specific option
            "permute_obs": True,      # environment-specific option
            "permute_actions": True,  # environment-specific option
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)


## Build The Model

This is the same model assembly pattern used by offline training: `NumericEmbedder` for row fields, `Qwen3Backbone` for sequence processing, and `DiscreteActionValueHead` for action values.

Because the online policy calls the model during rollout, the same model object is used in two modes: `eval()` for action selection and `train()` for optimizer updates.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
        },
    ],
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device=device, dtype=preferred_dtype(device))
print(model)


## Replay Buffer And Policy Contexts

Each environment stream writes to one `Datastore`; together they act as the replay buffer. Each environment also keeps a `contexts` list containing the recent rows used for action selection; `run_rollout` trims it to its `max_cache` bound at the start of each call.

The replay buffer is for training batches. The context list is for policy inference during rollout. Keeping them separate makes it clear which history is sampled for learning and which history conditions the next action.


In [ ]:
stores = [Datastore(name=name) for name in env.names]
contexts = [[] for _ in env.names]

# Online decode / replay: tokenizer (no augmenter/selector)
tokenizer = NumericTokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

transform = tokenizer

def pack_rows(rows: list[list[dict]]):
    """Ragged ``list[list[dict]]`` → ``TokenBatch`` via the shared transform."""
    steps, sids = [], []
    for i, seq in enumerate(rows):
        for step in seq:
            steps.append(transform(step))
            sids.append(i)
    inputs, _ = pack_token_batch(
        steps,
        sequence_ids=sids if steps else None,
        batch_size=len(rows),
        grouping_field="task_index",
    )
    return inputs


## Rollout Phase

A rollout collects `num_steps` lockstep env steps (callers pass `ROLLOUT_STEPS`) from the live environments. The loop follows the same shape each step:

1. Decide which environments should act greedily and which should explore randomly.
2. When any environment acts greedily, call `model(..., use_cache=True)` on the rows added since the last model call.
3. Convert predictions to actions with `model.get_action(...)`.
4. Step the `GroupEnv` with one input dictionary per environment.
5. Append each merged row to its datastore and context list.

`use_cache=True` returns a cache object that can be passed back into the next model call. This avoids recomputing the whole context on every action selection. Within a call the cache follows the same grow-then-rebuild policy as `09_inference.ipynb` — it grows incrementally until the cached span would exceed `max_cache`, then prefills from the latest `start_cache` rows — so it stays bounded even during long rollouts.

Row **contexts persist** across calls (trimmed to `max_cache` at the start); the **KV cache does not** and is rebuilt from those contexts on the first model call.


In [ ]:
def epsilon_for_env_step(*, env_step: int) -> float:
    """Return the epsilon-greedy exploration rate for a global env step."""
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    frac = min(env_step / EXPLORATION_ENDS, 1.0)
    return 1.0 - frac


In [ ]:
def run_rollout(*, model: Model, env, stores: list[Datastore], contexts: list[list], env_steps: int, num_steps: int, max_cache: int = SEQUENCE_LENGTH, start_cache: int | None = None) -> int:
    """Collect ``num_steps`` lockstep env steps and append rows to replay datastores.

    ``contexts`` persist across calls (trimmed to ``max_cache`` at the start).
    The KV cache is local to the call and follows the same grow-then-rebuild
    policy as ``09_inference.ipynb``: it grows incrementally until the cached span would
    exceed ``max_cache``, then prefills from the latest ``start_cache`` rows,
    so it stays bounded even within a long rollout.
    """
    env.metrics.clear()
    model.eval()
    max_cache, start_cache = resolve_cache_bounds(max_cache, start_cache)
    n = len(env.names)
    kv_cache = None
    cached_starts = np.zeros(n, dtype=np.int64)
    cached_ends = np.zeros(n, dtype=np.int64)
    context_start = np.zeros(n, dtype=np.int64)
    for i, c in enumerate(contexts):
        if len(c) > max_cache:
            contexts[i] = c[-max_cache:]
    for step in range(num_steps):
        epsilon = epsilon_for_env_step(env_step=env_steps)
        greedy = [bool(c) and torch.rand(1).item() >= epsilon for c in contexts]
        if any(greedy):
            ends = np.array([len(c) for c in contexts], dtype=np.int64)
            need_rebuild = cache_needs_rebuild(
                has_cache=kv_cache is not None,
                cached_starts=cached_starts,
                cached_ends=cached_ends,
                ends=ends,
                context_start=context_start,
                max_cache=max_cache,
                batch_complete=True,
            )
            with torch.no_grad():
                if need_rebuild:
                    starts = rebuild_starts(
                        ends=ends,
                        context_start=context_start,
                        start_cache=start_cache,
                        max_cache=max_cache,
                    )
                    batch = [contexts[i][int(starts[i]) : int(ends[i])] for i in range(n)]
                    predictions, kv_cache = model(pack_rows(batch), use_cache=True)
                    cached_starts = starts
                else:
                    batch = [contexts[i][int(cached_ends[i]) : int(ends[i])] for i in range(n)]
                    predictions, kv_cache = model(pack_rows(batch), cache=kv_cache, use_cache=True)
                cached_ends = ends.copy()
            actions = model.get_action(predictions, temperature=0.0, num_actions=MAX_ACTIONS)
        random_inputs = env.sample_random_input()
        inputs = [{'action': actions[i].cpu().numpy()} if greedy[i] else random_inputs[i] for i in range(n)]
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            row = {**inputs[i], **out}
            row.pop('info', None)
            stores[i].append(row)
            contexts[i].append(row)
        env_steps += len(outputs)
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return env_steps


## Training Phase

After a rollout, training uses the same API as offline training. `loader.refresh()` tells the dataloader to rescan the growing datastores and clear prefetched batches, then `run_train` runs `num_steps=TRAIN_STEPS` DQN updates. Delayed Q is `averager(averager_inputs)` (`tau_head`: delayed Q head on the packaged pooled representation). `averager.update()` Polyak-averages after the optimizer step.

`sequence_length` is a max window length — short stores are fine. `weight_mode="per_step"` samples in proportion to the amount of data in each store.


In [ ]:
loader = DataLoader(stores=stores, sequence_length=SEQUENCE_LENGTH, batch_size=BATCH_SIZE, weight_mode='per_step', num_workers=0, transform=transform)
objective = DqnObjective(
    gamma_step=1.0,                 # discount for ordinary next-step bootstrapping
    gamma_episode_terminal=1.0,     # discount when an episode ends normally
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,        # discount when a task ends normally
    gamma_task_truncated=0.0,
    grouping_field="task_index",
)
optimizer = AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)
averager = PolyakAverager(model, tau_head=POLYAK_TAU)

def run_train(
    *,
    model: Model,
    averager: PolyakAverager,
    optimizer: AdamW | AdamWFp32,
    objective: DqnObjective,
    loader: DataLoader,
    num_steps: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh replay and run ``num_steps`` optimizer updates."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()

        predictions, averager_inputs = model(inputs)
        delayed_predictions = averager(averager_inputs)
        loss, metrics = objective(objective_data.to(device), predictions, delayed_predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        averager.update()

    assert loss is not None
    return loss, metrics


## Run Online Training

The main loop runs `NUM_CYCLES` times. Each cycle calls `run_rollout(num_steps=ROLLOUT_STEPS)`, then `run_train(num_steps=TRAIN_STEPS)` once replay has at least `LEARNING_STARTS` rows. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
env_steps = 0
grad_steps = 0
for cycle in range(NUM_CYCLES):
    env_steps = run_rollout(model=model, env=env, stores=stores, contexts=contexts, env_steps=env_steps, num_steps=ROLLOUT_STEPS)
    print(f"cycle={cycle} rollout  env_step={env_steps}")
    if env_steps >= LEARNING_STARTS:
        loss, metrics = run_train(model=model, averager=averager, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
        grad_steps += TRAIN_STEPS
        print(f"cycle={cycle} train    loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}  grad_step={grad_steps}")
        if device.type == 'cuda':
            torch.cuda.empty_cache()
loader.close()
env.close()
print(f'Online training finished ({grad_steps} optimizer steps, {env_steps} env steps).')


## Push To The Hub

Run this cell to save the online-trained checkpoint. The inference notebook can load it later with `load_model`.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")